In [1]:
# Imports
import torch

In [ ]:
# Let's create a new tensor and use requires_grad=True
# grad true means it is doing back propagation and it will calculate the gradients for us 
#it works in the background and we can access it with .grad
#y = torch.rand(10)
a = torch.rand(10, requires_grad=True)
a


tensor([0.0602, 0.7383, 0.3560, 0.5724, 0.5468, 0.4104, 0.7333, 0.3355, 0.4787,
        0.4545], requires_grad=True)

In [20]:
# Let's build up a computation graph
# notice how this tensor had a special function called grad_fn

b = (a ** 2).mean()
b

tensor(0.2557, grad_fn=<MeanBackward0>)

In [21]:
# Lets compute one more tensor
c = b.mean()
c

tensor(0.2557, grad_fn=<MeanBackward0>)

In [22]:
# Since this is a scalar we can call backward on it
c.backward()
print(c)

tensor(0.2557, grad_fn=<MeanBackward0>)


In [ ]:
# Now we can see the gradients of a
print(a/10*2) # 1/10 comes from the mean and 2 comes from the power of 2
print(a.grad)

tensor([0.0120, 0.1477, 0.0712, 0.1145, 0.1094, 0.0821, 0.1467, 0.0671, 0.0957,
        0.0909], grad_fn=<MulBackward0>)
tensor([0.0120, 0.1477, 0.0712, 0.1145, 0.1094, 0.0821, 0.1467, 0.0671, 0.0957,
        0.0909])


In [ ]:

#zero when on the CPU but on the GPU it will show the amount of memory allocated
#thus far we are on CPU
torch.cuda.memory_allocated()

0

In [ ]:
#place the tensor on the GPU
#install for torch was CPU only so this will error out but if you have a GPU you can run this code

a = torch.rand(2**20, requires_grad=True, device="cuda")
torch.cuda.memory_allocated() / 1024.0**2

0.0

In [ ]:
#make computation graph on the GPU
#note how it doubles the memory usage because it is storing the intermediate values for backpropagation
b = torch.relu(a)
torch.cuda.memory_allocated() / 1024.0**2

0.0

In [30]:
#if we do more operations it will keep increasing the memory usage because it is storing the intermediate values for backpropagation

b = a
for _ in range(100):
    b = torch.relu(b)
torch.cuda.memory_allocated() / 1024.0**2

0.0

In [ ]:
#backward will free up the memory used for the intermediate values because it no longer needs them after it computes the gradients
#so the calculation pieces are lost (freed up) and just the solution is kept in memory
# 1+5+674+4+5+54+54+5+5+56 = 123456
# only 123456 is kept in memory after .backward() is called 
#thus calling .backward again won't work 
# 1+5+674+4+5+54+54+5+5+56.backward = 123456
# 123456.backward = error because the calculation pieces are lost (freed up) and just the solution is kept in memory
b.sum().backward() 
torch.cuda.memory_allocated() / 1024.0**2

In [ ]:
a.grad